<a href="https://colab.research.google.com/github/pjm9814/Econometrics/blob/metrics_spring_2026/MetricsII_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Econometrics II PS3
## Peter Marshall

In [ ]:
import pandas as pd
import numpy as np

(a) Write a code that generates i.i.d. samples of sizes n = 10, 50, 200 from that distribution and computes:
  - the least squares estimator for /beta/
  - the t-ratio for /beta_hat_1, LS/
  - the least squares residuals
Report /beta_hat_1, LS/ and /t_n/ for each of the three sample sizes.

In [ ]:
import statsmodels.api as sm

# Define the true parameters
beta_0 = 1
beta_1 = 1

sample_sizes = [10, 50, 200]

results_dict = {}

for n in sample_sizes:
    print(f"\n--- Sample Size: n = {n} ---")

    # Generate x_i once for each sample size and keep it fixed
    x = np.random.uniform(0, 2, n)

    # Generate epsilon_i for the current sample size
    epsilon = np.random.uniform(-1, 1, n)

    # Generate y_i
    y = beta_0 + beta_1 * x + epsilon

    # Add a constant to the independent variable for the intercept term
    X = sm.add_constant(x)

    # Perform OLS regression
    model = sm.OLS(y, X)
    results = model.fit()

    # (a) The least squares estimator for beta
    beta_0_hat_ls = results.params[0]
    beta_1_hat_ls = results.params[1]
    print(f"Least Squares Estimator β̂0_LS: {beta_0_hat_ls:.4f}")
    print(f"Least Squares Estimator β̂1_LS: {beta_1_hat_ls:.4f}")

    # Standard error for β̂1_LS
    se_beta_1_hat_ls = results.bse[1]

    # (b) The t-ratio for β̂1_LS
    # tn = (β̂1_LS - β_1_true) / s.e.(β̂1_LS)
    tn = (beta_1_hat_ls - beta_1) / se_beta_1_hat_ls
    print(f"t-ratio for β̂1_LS (tn): {tn:.4f}")

    # (c) The least squares residuals
    epsilon_hat = results.resid
    # print(f"Sample Residuals (first 5): {epsilon_hat[:5]}")

    results_dict[n] = {
        'beta_1_hat_ls': beta_1_hat_ls,
        'tn': tn,
        'residuals': epsilon_hat
    }

# Optional: Display all results at the end
print("\n--- Summary of Results ---")
for n, res in results_dict.items():
    print(f"Sample Size n = {n}: β̂1_LS = {res['beta_1_hat_ls']:.4f}, tn = {res['tn']:.4f}")


--- Sample Size: n = 10 ---
Least Squares Estimator β̂0_LS: 0.9929
Least Squares Estimator β̂1_LS: 1.1662
t-ratio for β̂1_LS (tn): 0.3968

--- Sample Size: n = 50 ---
Least Squares Estimator β̂0_LS: 1.0745
Least Squares Estimator β̂1_LS: 0.9731
t-ratio for β̂1_LS (tn): -0.1807

--- Sample Size: n = 200 ---
Least Squares Estimator β̂0_LS: 1.0475
Least Squares Estimator β̂1_LS: 0.9717
t-ratio for β̂1_LS (tn): -0.4176

--- Summary of Results ---
Sample Size n = 10: β̂1_LS = 1.1662, tn = 0.3968
Sample Size n = 50: β̂1_LS = 0.9731, tn = -0.1807
Sample Size n = 200: β̂1_LS = 0.9717, tn = -0.4176


(b) Write a code for drawing n times (with replacement) at random from the empirical distribution of the estimated residuals ˆϵ1, . . . , ϵˆn — that is, each residual is drawn with equal probability 1/n. Note this part only ask for the code, not the actual computation results. Figure out the correct input for this function would be sufficient.

In [ ]:
def bootstrap_residuals(residuals, n):
    """
    Draws n times (with replacement) at random from the empirical distribution
    of the estimated residuals.

    Args:
        residuals (np.ndarray): An array of estimated residuals (ˆϵ1, . . . , ϵˆn).
        n (int): The number of samples to draw (equivalent to the sample size).

    Returns:
        np.ndarray: An array of bootstrapped residuals.
    """
    # Each residual is drawn with equal probability 1/n
    bootstrapped_resids = np.random.choice(residuals, size=n, replace=True)
    return bootstrapped_resids

# Example of how you might call the function:
# Assuming 'epsilon_hat' is available from the previous cell (for the last sample size n=200)
# n_bootstrap = len(epsilon_hat) # Or any desired sample size for the bootstrap
# bootstrapped_residuals_example = bootstrap_residuals(epsilon_hat, n_bootstrap)
# print(f"First 5 bootstrapped residuals: {bootstrapped_residuals_example[:5]}")

(c) Implement the residual bootstrap 500 times. That is, keep xi fixed and use y*_i = /βˆ1,LSxi/ + ϵ*_i generate boostrap sample. Report the 95th percentiles of:
  • βˆ1,LS
  • tn
When computing the bootstrap t-statistic, subtract the original sample OLS estimate (not the true β1 = 1) in the numerator.

In [ ]:
num_bootstraps = 500

# Get the results from the original sample for n=200
# The variables `x`, `epsilon_hat`, `beta_1_hat_ls`, and `n` (which is 200)
# are available in the kernel from the previous execution for the last sample size.
# Let's ensure we use the correct n for the bootstrap, which is the original sample size.
original_n = n # This should be 200 from the last iteration of the loop in cell s0f8Db8Bd9UW
original_x = x # This is the x from the last iteration (n=200)
original_epsilon_hat = epsilon_hat # Residuals from the last iteration (n=200)
original_beta_1_hat_ls = beta_1_hat_ls # Beta_1_hat_ls from the last iteration (n=200)

bootstrapped_beta_1_hat_ls_values = []
bootstrapped_tn_values = []

for b in range(num_bootstraps):
    # 1. Generate bootstrapped residuals (ϵ*_i)
    epsilon_star = bootstrap_residuals(original_epsilon_hat, original_n)

    # 2. Generate bootstrap sample y*_i = βˆ1,LS * x_i + ϵ*_i
    # Use the original OLS estimate for beta_1 and the fixed x
    y_star = original_beta_1_hat_ls * original_x + epsilon_star

    # 3. Perform OLS regression on (y*_i, x_i)
    X_const = sm.add_constant(original_x) # Use the original fixed x with a constant
    bootstrap_model = sm.OLS(y_star, X_const)
    bootstrap_results = bootstrap_model.fit()

    # Extract bootstrapped βˆ*(b)1,LS and its standard error
    beta_1_hat_ls_bootstrap = bootstrap_results.params[1]
    se_beta_1_hat_ls_bootstrap = bootstrap_results.bse[1]

    # 4. Calculate the bootstrap t-statistic
    # t*(b)n = (βˆ*(b)1,LS - βˆ1,LS) / s.e.(βˆ*(b)1,LS)
    # Where βˆ1,LS is the original sample OLS estimate
    tn_bootstrap = (beta_1_hat_ls_bootstrap - original_beta_1_hat_ls) / se_beta_1_hat_ls_bootstrap

    # Store the results
    bootstrapped_beta_1_hat_ls_values.append(beta_1_hat_ls_bootstrap)
    bootstrapped_tn_values.append(tn_bootstrap)

# 5. Report the 95th percentiles
percentile_beta_1_hat_ls = np.percentile(bootstrapped_beta_1_hat_ls_values, 95)
percentile_tn = np.percentile(bootstrapped_tn_values, 95)

print(f"\n--- Residual Bootstrap Results (n={original_n}, {num_bootstraps} replications) ---")
print(f"Original β̂1_LS for n={original_n}: {original_beta_1_hat_ls:.4f}")
print(f"95th Percentile of Bootstrapped β̂1_LS: {percentile_beta_1_hat_ls:.4f}")
print(f"95th Percentile of Bootstrapped t_n: {percentile_tn:.4f}")


--- Residual Bootstrap Results (n=200, 500 replications) ---
Original β̂1_LS for n=200: 0.9717
95th Percentile of Bootstrapped β̂1_LS: 1.0820
95th Percentile of Bootstrapped t_n: 1.6542


(d) For each sample size, keep {xi} fixed, generate new residuals from  [−1, 1] (the true distribution of ϵi). Repeat this 500 times. Compute βˆ1LS and tn using 500 independent samples.

Use these to:

  • Simulate and report the 95th percentiles of their sampling distributions
  
This is different from the bootstrap method in that we use the true distribution of residuals to draw replications, which is not feasible with non-simulated data (that is, when we don’t know the true distribution of residuals).

In [ ]:
num_simulations = 500

# Define true parameters (already defined in previous cell, but for clarity)
beta_0 = 1
beta_1 = 1

# Get the sample sizes from the previous problem part
sample_sizes = [10, 50, 200]

print("\n--- Simulation Results (True Residual Distribution) ---")

for n in sample_sizes:
    print(f"\n--- Sample Size: n = {n} ---")

    # Generate x_i once for this sample size and keep it fixed
    # This ensures consistency with the problem statement 'keep xi fixed'
    # The 'x' variable from the previous cell's loop is only for n=200, so we regenerate for each n.
    x_fixed = np.random.uniform(0, 2, n)

    simulated_beta_1_hat_ls_values = []
    simulated_tn_values = []

    for _ in range(num_simulations):
        # Generate new residuals from the true distribution of epsilon_i
        epsilon_sim = np.random.uniform(-1, 1, n)

        # Generate simulated y_i using true parameters and fixed x
        y_sim = beta_0 + beta_1 * x_fixed + epsilon_sim

        # Add a constant to the independent variable for the intercept term
        X_sim = sm.add_constant(x_fixed)

        # Perform OLS regression
        model_sim = sm.OLS(y_sim, X_sim)
        results_sim = model_sim.fit()

        # Extract beta_1_hat_ls_sim
        beta_1_hat_ls_sim = results_sim.params[1]

        # Standard error for beta_1_hat_ls_sim
        se_beta_1_hat_ls_sim = results_sim.bse[1]

        # Calculate the t-ratio (using the true beta_1 = 1)
        tn_sim = (beta_1_hat_ls_sim - beta_1) / se_beta_1_hat_ls_sim

        simulated_beta_1_hat_ls_values.append(beta_1_hat_ls_sim)
        simulated_tn_values.append(tn_sim)

    # Report the 95th percentiles
    percentile_beta_1_hat_ls_sim = np.percentile(simulated_beta_1_hat_ls_values, 95)
    percentile_tn_sim = np.percentile(simulated_tn_values, 95)

    print(f"95th Percentile of Simulated β̂1_LS: {percentile_beta_1_hat_ls_sim:.4f}")
    print(f"95th Percentile of Simulated t_n: {percentile_tn_sim:.4f}")


--- Simulation Results (True Residual Distribution) ---

--- Sample Size: n = 10 ---
95th Percentile of Simulated β̂1_LS: 1.6503
95th Percentile of Simulated t_n: 2.1337

--- Sample Size: n = 50 ---
95th Percentile of Simulated β̂1_LS: 1.2784
95th Percentile of Simulated t_n: 1.8248

--- Sample Size: n = 200 ---
95th Percentile of Simulated β̂1_LS: 1.1161
95th Percentile of Simulated t_n: 1.6943


(e) Compare results from (c) and (d). What do you conclude about the bootstrap’s performance? How does it compare to the 95th percentile of tn computed from the t(n−2) distribution? (Recall that under normality of the errors, the OLS t-statistic follows a t(n−2) distribution exactly; this serves as an additional benchmark.)

In [ ]:
from scipy.stats import t

n_val = 200 # The sample size for which bootstrap was performed
df = n_val - 2 # Degrees of freedom for the t-distribution

t_dist_95_percentile = t.ppf(0.95, df)
print(f"Theoretical 95th Percentile of t({df}) distribution: {t_dist_95_percentile:.4f}")

print("\n--- Comparison of 95th Percentiles for n=200 ---")
print(f"Residual Bootstrap (c) - β̂1_LS: {percentile_beta_1_hat_ls:.4f}")
print(f"Simulation (d) - β̂1_LS:       {percentile_beta_1_hat_ls_sim:.4f}")
print(f"Residual Bootstrap (c) - t_n:  {percentile_tn:.4f}")
print(f"Simulation (d) - t_n:          {percentile_tn_sim:.4f}")
print(f"Theoretical t(n-2) - t_n:      {t_dist_95_percentile:.4f}")

Theoretical 95th Percentile of t(198) distribution: 1.6526

--- Comparison of 95th Percentiles for n=200 ---
Residual Bootstrap (c) - β̂1_LS: 1.0820
Simulation (d) - β̂1_LS:       1.1161
Residual Bootstrap (c) - t_n:  1.6542
Simulation (d) - t_n:          1.6943
Theoretical t(n-2) - t_n:      1.6526



1. **β̂1_LS Estimator:**
- The 95th percentile from the residual bootstrap (1.0820) is closer to the true beta_1 (1) than that from the simulation with true residuals (1.1161) but both are above 1. This suggests the residual bootstrap might capture the variability of the estimator reasonably well, though there's still a slight difference.
- Both values are greater than the true beta_1 = 1, indicating an upward skew in the distribution's upper tail, or a tendency for higher estimates at the 95th percentile.
2. **t_n Statistic:**
- The 95th percentile of the bootstrapped t_n (1.6542) is very close to the theoretical 95th percentile of the t(198) distribution (1.6525).
- The 95th percentile of the t_n from the simulation using true residuals (1.6943) is slightly higher than both the bootstrap and theoretical values.
- **Conclusion on Bootstrap Performance for t_n:** The residual bootstrap appears to perform quite well in approximating the sampling distribution of the t-statistic, especially when compared to the theoretical t-distribution. This is a positive indication of its ability to provide valid inferences even when the true error distribution is unknown."
- **Comparison to Theoretical t(n-2):** The bootstrap t-statistic's 95th percentile is notably close to the theoretical t-distribution's 95th percentile. This is a strong result, implying that the bootstrap provides a good approximation to the critical values or quantiles of the t-statistic's sampling distribution, even though the true errors are uniform, not normal. While OLS t-statistics follow a t(n-2) exactly *under normality of errors*, the bootstrap often works well even with non-normal errors due to its resampling nature, especially for larger sample sizes where the Central Limit Theorem helps the t-statistic distribution approximate a normal (and thus t) distribution."